In [ ]:

import json
import pennylane as qml
import pennylane.numpy as np
import scipy

H = np.array([[ 0.39488016,  0.04722628, -0.17943126, -0.03673282],
        [ 0.04722628,  0.37758558,  0.12997088,  0.17848188],
        [-0.17943126,  0.12997088,  0.53582574,  0.01115543],
        [-0.03673282,  0.17848188,  0.01115543,  0.29170851]])


U = scipy.linalg.expm(2j * np.pi * H)


def state_prep(params, wires):
    """
    Implements the state preparation circuit.

    Args:
        - params (np.array(float)): Angles [theta_1, theta_2, theta_3] parametrizing
        the RY rotations in the circuit.
        - wires (list): Labels for the circuit wires.
    Returns:
        - Does not return anything since it is a subcircuit.
    """
    qml.RY(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.RY(params[2], wires=wires[0])


dev = qml.device('lightning.qubit', wires = range(8))
 
@qml.qnode(dev)
def qpe_circuit(params):
    """
    Implements the QPE routine for the Unitary U, including initial state 
    preparation using the state_prep subcircuit.

    Args:
        - params (np.array(float)): Angles [theta_1, theta_2, theta_3] parametrizing
        the RY rotations in the state_prep circuit.
    Returns:
        - np.tensor(float): Computational basis probabilities in the estimation wires.
    """
    num_estimation_wires = 6
    e_wires = range(num_estimation_wires) # Wires 0-5
    t_wires = range(num_estimation_wires, num_estimation_wires + 2) # Wires 6-7 (for the 2-qubit Hamiltonian)

    # Prepare the initial state on target wires
    state_prep(params, t_wires)

    # Apply Hadamard gates to estimation wires
    for wire in e_wires:
        qml.Hadamard(wires=wire)

    # Apply controlled-U^k operations
    # The QPE circuit applies U^(2^(N-1-k)) controlled by estimation wire k
    for k in range(num_estimation_wires):
        power = 2**(num_estimation_wires - 1 - k)
        # Use qml.ctrl to apply controlled QubitUnitary
        qml.ctrl(qml.QubitUnitary, control=e_wires[k])(np.linalg.matrix_power(U, power), wires=t_wires)

    # Apply inverse QFT on estimation wires
    qml.adjoint(qml.QFT(wires=e_wires))

    # Return probabilities of estimation wires
    return qml.probs(wires=e_wires)


def compute_statistics(params):
    """
    Computes the phase and its uncertainty by postprocessing the results of the QPE circuit.

    Args:
        - params (np.array(float)): Angles [theta_1, theta_2, theta_3] parametrizing
        the RY rotations in the state_prep circuit.
    Returns:
        - mu (float): The phase calculated as a weighted average.
        - sigma (float): The uncertainty calculated as the standard deviation
        for the phase.
    """
    probs = qpe_circuit(params)
    num_estimation_wires = 6
    
    # Generate possible phases phi_i
    # The outcomes 'i' range from 0 to 2^N - 1.
    # Each outcome 'i' corresponds to a phase 'i / 2^N', which is the estimated eigenvalue.
    phases = np.array([i / (2**num_estimation_wires) for i in range(2**num_estimation_wires)])

    mu = np.sum(probs * phases)
    sigma = np.sqrt(np.sum(probs * (phases - mu)**2))

    return mu, sigma


def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    outs = list(compute_statistics(ins))
    
    return str(outs)


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.allclose(solution_output, expected_output, atol = 1e-4)
